# Heart Disease Data Analysis and Prediction
**Author:** Anushka  
**Dataset:** Heart Failure Prediction Dataset (`heart.csv`)  
**Source:** [Kaggle – fedesoriano/heart-failure-prediction](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction)

> ⚠️ **Academic Disclaimer:** This project is submitted as a college data-analytics assignment.  
> The machine-learning models developed here are **NOT** medical diagnostic tools and should **NOT** be used for clinical decision-making.


## 1. Introduction

Cardiovascular disease is one of the leading causes of death worldwide. Early identification of individuals at risk can significantly improve patient outcomes. This project performs exploratory data analysis (EDA) and applies basic machine-learning classification models on a publicly available heart disease dataset to understand patterns and evaluate predictive performance.

The dataset combines five publicly available heart disease datasets and contains **918 records** with **12 attributes** related to patient demographics, symptoms, and clinical measurements.


## 2. Problem Statement

Given clinical and demographic attributes of a patient, can we identify patterns that correlate with the presence of heart disease?  
The target variable `HeartDisease` is binary: `1` = Heart Disease present, `0` = No Heart Disease.


## 3. Objectives

1. Perform thorough exploratory data analysis on the dataset.
2. Identify key features that correlate with heart disease.
3. Clean and preprocess the data appropriately.
4. Train and evaluate classification models (Logistic Regression, Decision Tree, Random Forest).
5. Compare model performance using standard evaluation metrics.
6. Present findings in a clear, reproducible manner.


## 4. Import Libraries

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

# Output folder
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Libraries imported successfully.")


## 5. Load Dataset

In [ ]:
df = pd.read_csv('heart.csv')
print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")
df.head()


## 6. Dataset Overview

In [ ]:
print("Dataset Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())


In [ ]:
print("Data Types:")
print(df.dtypes)


In [ ]:
print("Descriptive Statistics:")
df.describe()


In [ ]:
# Categorical columns
categorical_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
numerical_cols   = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Numerical Columns :", numerical_cols)
print("Categorical Columns:", categorical_cols)
print()
for col in categorical_cols:
    print(f"  {col}: {df[col].unique().tolist()}")


## 7. Data Quality Check

In [ ]:
print("Missing Values:")
print(df.isnull().sum())
print()
print("Duplicate Rows:", df.duplicated().sum())


In [ ]:
# Check for clinically invalid zeros
print(f"RestingBP  == 0 : {(df['RestingBP']  == 0).sum()} rows")
print(f"Cholesterol == 0 : {(df['Cholesterol'] == 0).sum()} rows")


## 8. Data Cleaning

**Findings:**
- No missing values (NaN) found in the dataset.
- No duplicate rows found.
- **1 row** has `RestingBP = 0` — physiologically impossible; replaced with column median.
- **172 rows** have `Cholesterol = 0` — likely missing data encoded as zero; replaced with column median.


In [ ]:
df_clean = df.copy()

# Replace 0 RestingBP with median
df_clean['RestingBP'].replace(0, df_clean['RestingBP'].median(), inplace=True)

# Replace 0 Cholesterol with median
df_clean['Cholesterol'].replace(0, df_clean['Cholesterol'].median(), inplace=True)

print("Cleaning complete. Shape:", df_clean.shape)
print("RestingBP  zeros remaining:", (df_clean['RestingBP']  == 0).sum())
print("Cholesterol zeros remaining:", (df_clean['Cholesterol'] == 0).sum())


## 9. Exploratory Data Analysis

In [ ]:
# Target variable distribution
target_counts = df_clean['HeartDisease'].value_counts()
print("Target Variable Distribution:")
print(target_counts)
print(f"\nHeart Disease    (1): {target_counts[1]} ({target_counts[1]/len(df_clean)*100:.1f}%)")
print(f"No Heart Disease (0): {target_counts[0]} ({target_counts[0]/len(df_clean)*100:.1f}%)")


In [ ]:
# Age statistics by target
print("Age statistics by HeartDisease:")
df_clean.groupby('HeartDisease')['Age'].describe()


In [ ]:
# Sex vs Heart Disease crosstab
pd.crosstab(df_clean['Sex'], df_clean['HeartDisease'],
            rownames=['Sex'], colnames=['HeartDisease'])


In [ ]:
# ChestPainType vs Heart Disease
pd.crosstab(df_clean['ChestPainType'], df_clean['HeartDisease'],
            rownames=['ChestPainType'], colnames=['HeartDisease'])


## 10. Data Visualization

In [ ]:
# --- Target Distribution ---
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['No Heart Disease (0)', 'Heart Disease (1)'],
              [target_counts[0], target_counts[1]],
              color=['steelblue', 'tomato'], edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=11)
ax.set_title('Target Variable Distribution (HeartDisease)', fontsize=13)
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_target_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- Age Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_clean['Age'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
sns.boxplot(x='HeartDisease', y='Age', data=df_clean, ax=axes[1],
            palette=['steelblue', 'tomato'])
axes[1].set_title('Age vs Heart Disease')
axes[1].set_xlabel('HeartDisease (0=No, 1=Yes)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_age_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- Sex Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sex_counts = df_clean['Sex'].value_counts()
axes[0].bar(sex_counts.index, sex_counts.values,
            color=['steelblue', 'salmon'], edgecolor='white')
axes[0].set_title('Sex Distribution')
axes[0].set_ylabel('Count')

sex_disease = df_clean.groupby(['Sex', 'HeartDisease']).size().unstack()
sex_disease.plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'], edgecolor='white')
axes[1].set_title('Sex vs Heart Disease')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(['No Disease', 'Heart Disease'])
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_sex_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- Numerical Feature Distributions ---
num_features = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(num_features):
    axes[i].hist(df_clean[col], bins=25, color='steelblue', edgecolor='white')
    axes[i].set_title(f'{col} Distribution')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
axes[-1].set_visible(False)
plt.suptitle('Numerical Feature Distributions', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_numerical_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- Categorical Feature Distributions ---
cat_features = ['ChestPainType', 'RestingECG', 'ST_Slope']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(cat_features):
    counts = df_clean[col].value_counts()
    axes[i].bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    axes[i].set_title(f'{col} Distribution')
    axes[i].set_ylabel('Count')
plt.suptitle('Categorical Feature Distributions', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '05_categorical_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- Numerical Features vs Target ---
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(num_features):
    sns.boxplot(x='HeartDisease', y=col, data=df_clean, ax=axes[i],
                palette=['steelblue', 'tomato'])
    axes[i].set_title(f'{col} vs HeartDisease')
    axes[i].set_xlabel('HeartDisease (0=No, 1=Yes)')
axes[-1].set_visible(False)
plt.suptitle('Numerical Features vs Heart Disease', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '06_features_vs_target.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- Correlation Heatmap ---
df_enc = df_clean.copy()
le = LabelEncoder()
for col in categorical_cols:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_enc.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '07_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- ExerciseAngina & FastingBS vs Target ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, col in enumerate(['ExerciseAngina', 'FastingBS']):
    ct = pd.crosstab(df_clean[col], df_clean['HeartDisease'])
    ct.plot(kind='bar', ax=axes[i], color=['steelblue', 'tomato'], edgecolor='white')
    axes[i].set_title(f'{col} vs HeartDisease')
    axes[i].set_ylabel('Count')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=0)
    axes[i].legend(['No Disease', 'Heart Disease'])
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '08_exercise_fasting_vs_target.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- Chest Pain Type vs Target ---
fig, ax = plt.subplots(figsize=(7, 4))
ct = pd.crosstab(df_clean['ChestPainType'], df_clean['HeartDisease'])
ct.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'], edgecolor='white')
ax.set_title('Chest Pain Type vs Heart Disease')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(['No Disease', 'Heart Disease'])
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '09_chestpain_vs_target.png'), dpi=150, bbox_inches='tight')
plt.show()


## 11. Feature Engineering / Encoding

In [ ]:
df_model = df_clean.copy()
le = LabelEncoder()
for col in categorical_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

print("Encoded dataset (first 3 rows):")
df_model.head(3)


## 12. Train-Test Split

In [ ]:
X = df_model.drop('HeartDisease', axis=1)
y = df_model['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")
print(f"Features         : {X.shape[1]}")

# Feature scaling for Logistic Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print("Feature scaling applied for Logistic Regression.")


## 13. Machine Learning Models

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    Xtr = X_train_sc if name == 'Logistic Regression' else X_train
    Xte = X_test_sc  if name == 'Logistic Regression' else X_test

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)

    results[name] = {
        'Accuracy' : accuracy_score (y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall'   : recall_score   (y_test, y_pred, zero_division=0),
        'F1-Score' : f1_score       (y_test, y_pred, zero_division=0),
        'y_pred'   : y_pred,
        'cm'       : confusion_matrix(y_test, y_pred)
    }
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred,
                                 target_names=['No Disease', 'Heart Disease']))


## 14. Model Evaluation — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, res) in zip(axes, results.items()):
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Disease', 'Heart Disease'],
                yticklabels=['No Disease', 'Heart Disease'])
    ax.set_title(f'Confusion Matrix\n{name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '10_confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()


## 15. Model Comparison

In [ ]:
metrics     = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
model_names = list(results.keys())
values      = {m: [results[n][m] for n in model_names] for m in metrics}

x     = np.arange(len(model_names))
width = 0.2
fig, ax = plt.subplots(figsize=(11, 5))
for i, metric in enumerate(metrics):
    ax.bar(x + i * width, values[metric], width, label=metric, edgecolor='white')
ax.set_title('Model Performance Comparison', fontsize=14)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '11_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Feature Importance – Random Forest
rf_model    = models['Random Forest']
importances = rf_model.feature_importances_
feat_imp    = pd.Series(importances, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Random Forest — Feature Importances', fontsize=13)
ax.set_ylabel('Importance')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '12_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Summary table
results_df = pd.DataFrame({
    'Model'    : model_names,
    'Accuracy' : [results[n]['Accuracy']  for n in model_names],
    'Precision': [results[n]['Precision'] for n in model_names],
    'Recall'   : [results[n]['Recall']    for n in model_names],
    'F1-Score' : [results[n]['F1-Score']  for n in model_names],
})
results_df = results_df.set_index('Model').round(4)
print(results_df.to_string())


## 16. Results and Findings

### Model Performance Summary

| Model | Accuracy | Precision | Recall | F1-Score |
|---|---|---|---|---|
| Logistic Regression | 0.8696 | 0.8482 | 0.9314 | 0.8879 |
| Decision Tree | 0.8098 | 0.8252 | 0.8333 | 0.8293 |
| **Random Forest** | **0.8750** | **0.8762** | **0.9020** | **0.8889** |

### Key Findings from EDA

1. **Dataset Balance:** The dataset is slightly imbalanced — 55.3% heart disease vs 44.7% no heart disease.
2. **Age:** Patients with heart disease tend to be older (mean ~56) vs those without (mean ~51).
3. **Sex:** Males make up ~79% of the dataset; ASY chest pain is more prevalent among males with heart disease.
4. **Chest Pain Type:** Asymptomatic (ASY) chest pain is strongly associated with heart disease presence.
5. **ST_Slope:** Flat or downward ST_Slope strongly indicates heart disease presence.
6. **Exercise Angina:** Patients with exercise-induced angina have a much higher heart disease rate.
7. **Oldpeak:** Higher ST depression (Oldpeak) correlates with heart disease.
8. **Cholesterol:** 172 rows had zero cholesterol (replaced with median during cleaning).


## 17. Conclusion

This project successfully performed end-to-end data analysis and machine learning on the Heart Failure Prediction dataset.

- Random Forest achieved the best accuracy of **87.50%** with F1-score of **0.8889**.
- Logistic Regression performed comparably with 86.96% accuracy.
- Key risk factors identified include: ST_Slope, ExerciseAngina, ChestPainType, Oldpeak, and MaxHR.

The analysis confirms that clinical measurements combined with demographic data can effectively predict heart disease presence using standard classification algorithms.


## 18. Limitations

1. The dataset contains 172 zero-cholesterol values that required imputation, which may introduce some bias.
2. The dataset is predominantly male (~79%), limiting generalizability to female populations.
3. No hyperparameter tuning was performed in this baseline study.
4. The models are not validated on an external independent dataset.
5. This analysis is for academic purposes only and is **not a medical diagnostic tool**.


## 19. Future Scope

1. Apply hyperparameter tuning (GridSearchCV / RandomizedSearchCV) to improve model performance.
2. Explore advanced models such as XGBoost, SVM, and Neural Networks.
3. Address class imbalance using SMOTE or other resampling techniques.
4. Perform cross-validation for more robust evaluation.
5. Develop an interactive dashboard using Streamlit or Power BI for visualization.
6. Extend analysis to include feature selection techniques (RFE, SHAP values).
